# Phase 12 ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬ÃƒÂ¢Ã¢â€šÂ¬Ã‚Â Controlled XGBoost V2 deployment

This notebook validates the stored Phase 11 decision, deploys the existing V2 artifact through the FastAPI application, exercises restart-based V1 rollback, and writes auditable JSON/CSV results. It does not retrain models or modify frozen-phase artifacts.

## Scope and safety

The active model is selected at process startup with `FRAUD_ACTIVE_MODEL_VERSION` (`v2` by default). The notebook uses isolated local API processes and removes only its own temporary prediction log.

In [ ]:

from __future__ import annotations

import csv
import hashlib
import json
import os
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
from xgboost import XGBClassifier

# ============================================================
# PHASE 12: V2 DEPLOYMENT VALIDATION
# Read-only model validation + isolated Kafka/Spark integration.
# No retraining. No changes to frozen V1/V2 model artifacts.
# ============================================================

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "models").is_dir():
    ROOT = ROOT.parent

if not (ROOT / "models").is_dir():
    raise RuntimeError("Could not locate repository root containing models/.")

sys.path.insert(0, str(ROOT))

REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_JSON = REPORT_DIR / "model_v2_deployment_report.json"
REPORT_CSV = REPORT_DIR / "model_deployment_validation.csv"

ARTIFACTS = {
    "v1_model": ROOT / "models" / "xgboost_model_v1.json",
    "v1_metadata": ROOT / "models" / "model_v1_metadata.json",
    "v1_threshold": ROOT / "models" / "model_v1_threshold.json",
    "v2_model": ROOT / "models" / "xgboost_model_v2.json",
    "v2_metadata": ROOT / "models" / "model_v2_metadata.json",
    "v2_threshold": ROOT / "models" / "model_v2_threshold.json",
    "phase11_training_report": REPORT_DIR / "model_v2_training_report.json",
    "phase11_comparison": REPORT_DIR / "model_v1_vs_v2_comparison.json",
}

TOLERANCE = 1e-6
started_at = datetime.now(timezone.utc).isoformat()
tests = []


# ============================================================
# HELPERS
# ============================================================

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_json(path: Path) -> dict:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def record(name, status, expected, actual, details):
    tests.append({
        "test_name": name,
        "status": status,
        "expected": expected,
        "actual": actual,
        "details": details,
    })


def check(name, passed, expected, actual, details):
    record(
        name,
        "PASS" if passed else "FAIL",
        expected,
        actual,
        details,
    )


def stop_process(process):
    if process is None or process.poll() is not None:
        return

    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=10)


def request_json(url, payload=None, method=None):
    body = None if payload is None else json.dumps(payload).encode("utf-8")

    request = urllib.request.Request(
        url,
        data=body,
        headers={"Content-Type": "application/json"},
        method=method or ("POST" if payload is not None else "GET"),
    )

    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            raw = response.read().decode("utf-8")
            return response.status, json.loads(raw)

    except urllib.error.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        try:
            body = json.loads(raw)
        except json.JSONDecodeError:
            body = {"raw_response": raw}
        return exc.code, body

    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        return 0, {"error": str(exc)}


def probability(response):
    value = response.get("fraud_probability") if isinstance(response, dict) else None
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    return None


def run_api(version, port):
    env = os.environ.copy()
    env["FRAUD_ACTIVE_MODEL_VERSION"] = version
    env["PYTHONPATH"] = str(ROOT) + os.pathsep + env.get("PYTHONPATH", "")

    process = subprocess.Popen(
        [
            sys.executable, "-m", "uvicorn",
            "src.api.app:app",
            "--host", "127.0.0.1",
            "--port", str(port),
        ],
        cwd=str(ROOT),
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    base = f"http://127.0.0.1:{port}"

    for _ in range(60):
        if process.poll() is not None:
            raise RuntimeError(
                f"API {version} exited during startup on port {port}."
            )

        status, health = request_json(base + "/health")
        if status == 200:
            return process, base, health

        time.sleep(0.25)

    stop_process(process)
    raise RuntimeError(f"API {version} failed to become healthy on port {port}.")


# ============================================================
# 1. ARTIFACT PRESENCE, JSON, AND PHASE 11 DECISION
# ============================================================

missing = [
    name for name, path in ARTIFACTS.items()
    if not path.is_file() or path.stat().st_size == 0
]

if missing:
    raise FileNotFoundError(
        "Missing artifacts or Phase 11 reports: " + ", ".join(missing)
    )

hashes_before = {
    name: sha256(path)
    for name, path in ARTIFACTS.items()
}

json_values = {
    name: read_json(path)
    for name, path in ARTIFACTS.items()
    if path.suffix == ".json"
}

v1_metadata = json_values["v1_metadata"]
v2_metadata = json_values["v2_metadata"]
v1_threshold = json_values["v1_threshold"]
v2_threshold = json_values["v2_threshold"]
comparison = json_values["phase11_comparison"]
training = json_values["phase11_training_report"]

check(
    "artifact_presence_and_json",
    not missing,
    "All frozen artifacts and Phase 11 reports exist and parse",
    {"missing": missing, "parsed_json_files": len(json_values)},
    "Read-only artifact and JSON validation.",
)

decision = comparison.get("decision")

check(
    "phase11_decision",
    decision == "V2_CANDIDATE_FOR_DEPLOYMENT",
    "V2_CANDIDATE_FOR_DEPLOYMENT",
    decision,
    "Stored Phase 11 decision.",
)


# ============================================================
# 2. FEATURE CONTRACT AND THRESHOLD
# ============================================================

v1_features = v1_metadata.get("feature_names", [])
v2_features = v2_metadata.get("feature_names", [])

contract_ok = (
    len(v1_features) == 34
    and v1_features == v2_features
    and len(set(v2_features)) == 34
)

check(
    "feature_contract",
    contract_ok,
    {"count": 34, "same_order": True},
    {
        "v1_count": len(v1_features),
        "v2_count": len(v2_features),
        "same_order": v1_features == v2_features,
    },
    "Exact ordered feature contract from frozen metadata.",
)

threshold = v2_threshold.get("selected_threshold")

threshold_ok = (
    isinstance(threshold, (int, float))
    and not isinstance(threshold, bool)
    and 0 <= threshold <= 1
    and threshold == v2_metadata.get("selected_classification_threshold")
)

check(
    "v2_threshold_contract",
    threshold_ok,
    "V2 threshold matches metadata and lies in [0, 1]",
    threshold,
    "V2 threshold artifact validation.",
)

# Do not continue into prediction tests with a broken feature/threshold contract.
if not contract_ok or not threshold_ok:
    raise RuntimeError(
        "Feature or threshold contract failed. Resolve the inconsistency "
        "before continuing Phase 12."
    )


# ============================================================
# 3. LOAD MODELS WITHOUT RETRAINING
# ============================================================

from src.api import app as api

v1_model = XGBClassifier()
v1_model.load_model(str(ARTIFACTS["v1_model"]))

v2_model = XGBClassifier()
v2_model.load_model(str(ARTIFACTS["v2_model"]))

check(
    "model_artifact_loading",
    True,
    "Both frozen XGBClassifier artifacts load",
    {"v1_loaded": True, "v2_loaded": True},
    "Both models loaded without training or artifact writes.",
)

model_contract_ok = (
    list(v1_model.get_booster().feature_names or []) == v1_features
    and list(v2_model.get_booster().feature_names or []) == v2_features
)

check(
    "model_feature_contract",
    model_contract_ok,
    "Loaded models expose metadata feature order",
    model_contract_ok,
    "Booster feature names compared with frozen metadata.",
)


# ============================================================
# 4. DIRECT V1/V2 INFERENCE
# ============================================================

txn_data = {
    "step": 500,
    "type": "TRANSFER",
    "amount": 1500.0,
    "nameOrig": "C1234567890",
    "oldbalanceOrg": 5000.0,
    "nameDest": "C9876543210",
    "oldbalanceDest": 0.0,
    "isFlaggedFraud": 0,
}

api.account_state.reset()
txn = api.TransactionRequest(**txn_data)
features = api.build_feature_vector(txn, update_state=False)

direct_v1_probability = float(v1_model.predict_proba(features)[0, 1])
direct_v2_probability = float(v2_model.predict_proba(features)[0, 1])
direct_v2_label = int(direct_v2_probability >= threshold)

direct_ok = (
    features.columns.tolist() == v2_features
    and np.isfinite(direct_v1_probability)
    and np.isfinite(direct_v2_probability)
    and 0 <= direct_v1_probability <= 1
    and 0 <= direct_v2_probability <= 1
)

check(
    "direct_v1_v2_compatibility",
    direct_ok,
    "Both models accept the same exact 34-feature input",
    {
        "feature_count": len(features.columns),
        "columns_match": features.columns.tolist() == v2_features,
        "v1_probability": direct_v1_probability,
        "v2_probability": direct_v2_probability,
        "v2_label": direct_v2_label,
    },
    "Direct inference through the frozen API feature builder.",
)


# ============================================================
# 5. V2 API HEALTH, PREDICTION PARITY, AND VALIDATION
# ============================================================

v2_process = None
api_prediction = {}
api_repeat = {}
health = {}
first_probability = None
api_base = None

try:
    v2_process, api_base, health = run_api("v2", 8765)

    health_ok = (
        health.get("model_version") == "V2"
        and health.get("model_loaded") is True
        and health.get("feature_count") == 34
        and health.get("threshold") == threshold
    )

    check(
        "api_health_v2",
        health_ok,
        {
            "model_version": "V2",
            "feature_count": 34,
            "threshold": threshold,
        },
        health,
        "Fresh V2 process health response.",
    )

    status, api_prediction = request_json(api_base + "/predict", txn_data)

    # Reset process-local state and replay the same transaction.
    reset_status, reset_body = request_json(
        api_base + "/state/reset", method="POST"
    )
    repeat_status, api_repeat = request_json(
        api_base + "/predict", txn_data
    )

    first_probability = probability(api_prediction)
    repeat_probability = probability(api_repeat)

    difference = (
        abs(first_probability - direct_v2_probability)
        if first_probability is not None else None
    )
    repeat_difference = (
        abs(first_probability - repeat_probability)
        if first_probability is not None and repeat_probability is not None
        else None
    )

    parity_ok = (
        status == 200
        and reset_status == 200
        and repeat_status == 200
        and difference is not None
        and repeat_difference is not None
        and difference <= TOLERANCE
        and repeat_difference <= TOLERANCE
        and api_prediction.get("prediction") == direct_v2_label
        and api_repeat.get("prediction") == api_prediction.get("prediction")
    )

    check(
        "api_v2_prediction_parity",
        parity_ok,
        {
            "http_status": 200,
            "tolerance": TOLERANCE,
            "deterministic_after_state_reset": True,
        },
        {
            "status": status,
            "reset_status": reset_status,
            "repeat_status": repeat_status,
            "direct_probability": direct_v2_probability,
            "api_probability": first_probability,
            "repeat_probability": repeat_probability,
            "direct_difference": difference,
            "repeat_difference": repeat_difference,
            "api_prediction": api_prediction,
            "repeat_prediction": api_repeat,
        },
        "Strict direct-versus-API parity; no tolerance relaxation.",
    )

    # Ground-truth leakage must be rejected.
    leakage_status, leakage_body = request_json(
        api_base + "/predict",
        {**txn_data, "isFraud": 0},
    )

    check(
        "ground_truth_rejection",
        leakage_status == 422,
        422,
        {"status": leakage_status, "body": leakage_body},
        "API must reject the isFraud ground-truth field.",
    )

    # Missing required field must be rejected.
    malformed = {
        key: value for key, value in txn_data.items()
        if key != "amount"
    }
    malformed_status, malformed_body = request_json(
        api_base + "/predict", malformed
    )

    check(
        "request_validation",
        malformed_status == 422,
        422,
        {"status": malformed_status, "body": malformed_body},
        "API must reject requests missing required fields.",
    )

    # Simple sequential latency smoke test; not a load test.
    timings = []
    errors = []

    for _ in range(5):
        reset_code, _ = request_json(
            api_base + "/state/reset", method="POST"
        )
        began = time.perf_counter()
        code, body = request_json(api_base + "/predict", txn_data)
        timings.append((time.perf_counter() - began) * 1000)

        if reset_code != 200 or code != 200 or probability(body) is None:
            errors.append({"reset_status": reset_code, "status": code, "body": body})

    performance = {
        "total_requests": 5,
        "successful_requests": 5 - len(errors),
        "failed_requests": len(errors),
        "median_latency_ms": float(np.median(timings)),
        "minimum_latency_ms": float(np.min(timings)),
        "maximum_latency_ms": float(np.max(timings)),
        "method": "Five sequential requests; startup excluded",
        "errors": errors,
    }

    check(
        "api_performance",
        not errors,
        {"successful_requests": 5, "failed_requests": 0},
        performance,
        "Functional latency smoke test, not a production load benchmark.",
    )

    # Independent V2 process/cold-start parity.
    fresh_process = None

    try:
        fresh_process, fresh_base, fresh_health = run_api("v2", 8768)
        fresh_reset, _ = request_json(
            fresh_base + "/state/reset", method="POST"
        )
        fresh_status, fresh_response = request_json(
            fresh_base + "/predict", txn_data
        )

        fresh_probability = probability(fresh_response)
        fresh_difference = (
            abs(direct_v2_probability - fresh_probability)
            if fresh_probability is not None else None
        )

        fresh_ok = (
            fresh_health.get("model_version") == "V2"
            and fresh_reset == 200
            and fresh_status == 200
            and fresh_difference is not None
            and fresh_difference <= TOLERANCE
        )

        check(
            "fresh_process_parity",
            fresh_ok,
            {"status": 200, "tolerance": TOLERANCE},
            {
                "health": fresh_health,
                "reset_status": fresh_reset,
                "status": fresh_status,
                "direct_probability": direct_v2_probability,
                "fresh_probability": fresh_probability,
                "difference": fresh_difference,
            },
            "Independent V2 process with fresh account state.",
        )

    finally:
        stop_process(fresh_process)

except Exception as exc:
    record(
        "api_v2_runtime",
        "FAIL",
        "V2 API starts and validation requests complete",
        None,
        f"{type(exc).__name__}: {exc}",
    )

finally:
    stop_process(v2_process)


# ============================================================
# 6. V1 ROLLBACK AND V2 RESTORATION
# ============================================================

rollback_process = None

try:
    rollback_process, rollback_base, rollback_health = run_api("v1", 8766)
    rollback_reset, _ = request_json(
        rollback_base + "/state/reset", method="POST"
    )
    rollback_status, rollback_response = request_json(
        rollback_base + "/predict", txn_data
    )

    rollback_ok = (
        rollback_health.get("model_version") == "V1"
        and rollback_reset == 200
        and rollback_status == 200
        and probability(rollback_response) is not None
    )

    check(
        "restart_rollback_v1",
        rollback_ok,
        {"model_version": "V1", "status": 200},
        {
            "health": rollback_health,
            "reset_status": rollback_reset,
            "status": rollback_status,
            "prediction": rollback_response,
        },
        "Fresh-process V1 rollback test.",
    )

except Exception as exc:
    record(
        "restart_rollback_v1",
        "FAIL",
        "Fresh V1 process starts and predicts",
        None,
        f"{type(exc).__name__}: {exc}",
    )

finally:
    stop_process(rollback_process)


restore_process = None

try:
    restore_process, restore_base, restore_health = run_api("v2", 8767)
    restore_reset, _ = request_json(
        restore_base + "/state/reset", method="POST"
    )
    restore_status, restore_response = request_json(
        restore_base + "/predict", txn_data
    )

    restore_ok = (
        restore_health.get("model_version") == "V2"
        and restore_reset == 200
        and restore_status == 200
        and probability(restore_response) is not None
    )

    check(
        "restore_v2",
        restore_ok,
        {"model_version": "V2", "status": 200},
        {
            "health": restore_health,
            "reset_status": restore_reset,
            "status": restore_status,
            "prediction": restore_response,
        },
        "Fresh-process V2 restoration after V1 rollback test.",
    )

except Exception as exc:
    record(
        "restore_v2",
        "FAIL",
        "Fresh V2 process starts and predicts",
        None,
        f"{type(exc).__name__}: {exc}",
    )

finally:
    stop_process(restore_process)


# ============================================================
# 7. KAFKA SCHEMA AND SPARK SOURCE COMPATIBILITY
# ============================================================

try:
    from src.streaming.kafka_producer import KAFKA_MESSAGE_FIELDS

    kafka_schema_ok = (
        len(KAFKA_MESSAGE_FIELDS) == 10
        and "isFraud" not in KAFKA_MESSAGE_FIELDS
    )

    check(
        "kafka_schema_compatibility",
        kafka_schema_ok,
        "Ten transaction-time fields; no isFraud",
        list(KAFKA_MESSAGE_FIELDS),
        "Validates the existing producer schema.",
    )

except Exception as exc:
    record(
        "kafka_schema_compatibility",
        "FAIL",
        "Existing Kafka schema is readable",
        None,
        f"{type(exc).__name__}: {exc}",
    )


spark_source = ROOT / "src" / "streaming" / "spark_streaming.py"
spark_text = (
    spark_source.read_text(encoding="utf-8")
    if spark_source.is_file() else ""
)

spark_ok = (
    spark_source.is_file()
    and "FINAL_FEATURES" in spark_text
    and "isFraud" in spark_text
    and "startingOffsets" in spark_text
)

check(
    "spark_feature_contract_compatibility",
    spark_ok,
    "Spark source references the feature contract and Kafka offsets",
    {
        "source_exists": spark_source.is_file(),
        "final_features_marker": "FINAL_FEATURES" in spark_text,
        "isFraud_marker": "isFraud" in spark_text,
        "starting_offsets_marker": "startingOffsets" in spark_text,
    },
    "Offline source inspection; not a substitute for integration testing.",
)


# ============================================================
# 8. REAL V2 KAFKA -> SPARK END-TO-END TEST
# Use the verified integration runner instead of the old
# KafkaAdminClient setup that used an unsupported argument.
# ============================================================

integration_status = "BLOCKED"
integration_details = ""
integration_script = (
    ROOT / "tests" / "phase12_v2_kafka_spark_integration.py"
)

integration_actual = {
    "executed": False,
    "model_version": None,
    "topic": None,
    "producer_transport": "kafka",
    "messages_sent": 0,
    "prediction_rows": 0,
    "checkpoint_created": False,
}

# Only accept evidence generated by this execution.
evidence_pattern = "phase12_v2_integration_evidence_*.json"
existing_evidence = {
    p.resolve() for p in REPORT_DIR.glob(evidence_pattern)
}

try:
    if not integration_script.is_file():
        integration_details = (
            f"Integration runner not found: {integration_script}"
        )
        integration_actual["error"] = integration_details

    else:
        child_env = os.environ.copy()
        child_env["FRAUD_ACTIVE_MODEL_VERSION"] = "v2"
        child_env["PYTHONPATH"] = (
            str(ROOT)
            + os.pathsep
            + child_env.get("PYTHONPATH", "")
        )

        integration_actual["executed"] = True

        result = subprocess.run(
            [sys.executable, "-u", str(integration_script)],
            cwd=str(ROOT),
            env=child_env,
            capture_output=True,
            text=True,
            timeout=240,
            check=False,
        )

        integration_actual["runner_return_code"] = result.returncode
        integration_actual["runner_stdout"] = result.stdout[-8000:]
        integration_actual["runner_stderr"] = result.stderr[-8000:]

        new_evidence = [
            p for p in REPORT_DIR.glob(evidence_pattern)
            if p.resolve() not in existing_evidence
        ]

        if not new_evidence:
            integration_status = (
                "FAIL" if result.returncode != 0 else "BLOCKED"
            )
            integration_details = (
                "No new integration evidence JSON was generated. "
                "Inspect runner_stdout and runner_stderr."
            )
            integration_actual["error"] = integration_details

        else:
            evidence_path = max(
                new_evidence,
                key=lambda p: p.stat().st_mtime,
            )

            with evidence_path.open(
                "r", encoding="utf-8"
            ) as handle:
                evidence = json.load(handle)

            integration_actual["evidence_file"] = str(
                evidence_path.relative_to(ROOT)
            )

            checks = evidence.get("checks", {})
            prediction = evidence.get("prediction", {})

            # These names and values are validated against the
            # actual evidence, not inferred from the process exit code.
            required_checks = [
                "kafka_publish",
                "prediction",
                "threshold",
                "risk_level",
                "checkpoint",
                "feature_contract",
                "output",
            ]

            check_statuses = {
                name: checks.get(name, {}).get("status")
                for name in required_checks
            }

            try:
                pred_probability = float(
                    prediction["fraud_probability"]
                )
                pred_label = int(prediction["predicted_label"])

                probability_ok = (
                    np.isfinite(pred_probability)
                    and 0.0 <= pred_probability <= 1.0
                )
                threshold_ok = (
                    probability_ok
                    and pred_label
                    == int(pred_probability >= float(threshold))
                )
            except (KeyError, TypeError, ValueError):
                pred_probability = None
                pred_label = None
                probability_ok = False
                threshold_ok = False

            model_version = str(
                prediction.get("model_version", "")
            ).upper()

            evidence_threshold = evidence.get("threshold")
            threshold_matches = (
                isinstance(evidence_threshold, (int, float))
                and abs(float(evidence_threshold) - float(threshold))
                <= 1e-12
            )

            output_rel = evidence.get("result_file")
            checkpoint_rel = evidence.get("checkpoint_path")

            output_path = (
                ROOT / output_rel if output_rel else None
            )
            checkpoint_path = (
                ROOT / checkpoint_rel if checkpoint_rel else None
            )

            csv_rows = []
            if output_path is not None and output_path.is_file():
                try:
                    with output_path.open(
                        "r",
                        newline="",
                        encoding="utf-8-sig",
                    ) as handle:
                        csv_rows = list(csv.DictReader(handle))
                except (OSError, csv.Error):
                    csv_rows = []

            checkpoint_ok = (
                checkpoint_path is not None
                and checkpoint_path.is_dir()
                and checks.get("checkpoint", {}).get("status") == "PASS"
            )

            topic = evidence.get("topic")
            topic_ok = (
                isinstance(topic, str)
                and topic.startswith("phase12-v2-integration-")
            )

            checks_passed = all(
                value == "PASS"
                for value in check_statuses.values()
            )

            all_conditions_pass = all([
                result.returncode == 0,
                "PHASE12_V2_KAFKA_SPARK_INTEGRATION: PASS"
                in result.stdout,
                evidence.get("status") == "PASS",
                checks_passed,
                model_version == "V2",
                threshold_matches,
                probability_ok,
                threshold_ok,
                len(csv_rows) == 1,
                evidence.get("prediction_rows") == 1,
                evidence.get("messages_sent") == 1,
                checkpoint_ok,
                topic_ok,
            ])

            integration_actual.update({
                "model_version": model_version,
                "topic": topic,
                "messages_sent": evidence.get("messages_sent", 0),
                "prediction_rows": len(csv_rows),
                "fraud_probability": pred_probability,
                "predicted_label": pred_label,
                "expected_threshold": threshold,
                "threshold_label_verified": threshold_ok,
                "threshold_matches_notebook": threshold_matches,
                "checkpoint_created": checkpoint_ok,
                "checkpoint_path": (
                    str(checkpoint_path.relative_to(ROOT))
                    if checkpoint_path is not None
                    else None
                ),
                "result_file": (
                    str(output_path.relative_to(ROOT))
                    if output_path is not None
                    else None
                ),
                "evidence_checks": check_statuses,
            })

            if all_conditions_pass:
                integration_status = "PASS"
                integration_details = (
                    "Verified real Kafka-to-Spark integration passed. "
                    "One message produced one V2 prediction, and the "
                    "threshold, model version, feature contract, risk "
                    "level, output, and checkpoint checks passed. "
                    "This does not by itself establish production readiness."
                )
            else:
                integration_status = "FAIL"
                integration_details = (
                    "Integration evidence was produced, but one or more "
                    "validation conditions failed. Inspect integration_actual, "
                    "the evidence JSON, and the runner logs."
                )

except subprocess.TimeoutExpired:
    integration_status = "FAIL"
    integration_details = (
        "Integration runner exceeded the 240-second timeout."
    )
    integration_actual["error"] = integration_details

except Exception as exc:
    integration_status = "FAIL"
    integration_details = f"{type(exc).__name__}: {exc}"
    integration_actual["error"] = integration_details

record(
    "kafka_spark_end_to_end",
    integration_status,
    {
        "broker_backed": True,
        "messages_sent": 1,
        "prediction_model_version": "V2",
        "threshold_label_verified": True,
        "checkpoint_created": True,
    },
    integration_actual,
    integration_details,
)

print("\nV2 KAFKA -> SPARK TEST")
print("Status:", integration_status)
print("Topic:", integration_actual.get("topic"))
print("Model version:", integration_actual.get("model_version"))
print("Messages sent:", integration_actual.get("messages_sent"))
print("Prediction rows:", integration_actual.get("prediction_rows"))
print("Checkpoint:", integration_actual.get("checkpoint_created"))
print("Evidence:", integration_actual.get("evidence_file"))
print("Details:", integration_details)


# ============================================================
# 9. FROZEN ARTIFACT INTEGRITY
# ============================================================

hashes_after = {
    name: sha256(path)
    for name, path in ARTIFACTS.items()
}

integrity_ok = hashes_before == hashes_after

check(
    "artifact_integrity",
    integrity_ok,
    "Frozen model, metadata, threshold, and Phase 11 report hashes unchanged",
    {
        "unchanged": integrity_ok,
        "before": hashes_before,
        "after": hashes_after,
    },
    "SHA-256 comparison before and after validation.",
)


# ============================================================
# 10. FINAL STATUS AND REPORTS
# ============================================================

failed = [t for t in tests if t["status"] == "FAIL"]
blocked = [t for t in tests if t["status"] == "BLOCKED"]
passed = [t for t in tests if t["status"] == "PASS"]

# A validation pass is not the same thing as production deployment.
final_status = (
    "V2_VALIDATION_FAILED"
    if failed
    else "V2_VALIDATION_BLOCKED"
    if blocked
    else "V2_VALIDATION_PASSED"
)

report = {
    "phase": 12,
    "final_status": final_status,
    "started_at_utc": started_at,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase11_decision": decision,
    "direct_v1_probability": direct_v1_probability,
    "direct_v2_probability": direct_v2_probability,
    "api_v2_probability": probability(api_prediction),
    "v2_restoration_validation": next(
        (
            t["status"] for t in tests
            if t["test_name"] == "restore_v2"
        ),
        "NOT_RUN",
    ),
    "tests": tests,
    "summary": {
        "passed": len(passed),
        "failed": len(failed),
        "blocked": len(blocked),
        "total": len(tests),
    },
    "artifact_hashes_before": hashes_before,
    "artifact_hashes_after": hashes_after,
    "notes": [
        "No model training was performed.",
        "Frozen model, metadata, threshold, and Phase 11 report artifacts "
        "were SHA-256 checked before and after validation.",
        "Kafka/Spark integration status is based on the actual bounded test.",
        "A passing validation report is not, by itself, proof of production deployment.",
        "The API latency check is a small sequential smoke test, not a load test.",
    ],
}

REPORT_JSON.write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8",
)

with REPORT_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["test_name", "status", "expected", "actual", "details"],
    )
    writer.writeheader()

    for test in tests:
        writer.writerow({
            key: (
                json.dumps(test[key], sort_keys=True, default=str)
                if isinstance(test[key], (dict, list))
                else test[key]
            )
            for key in writer.fieldnames
        })

print("\n" + "=" * 60)
print("PHASE 12 VALIDATION COMPLETE")
print("=" * 60)
print("Final status:", final_status)
print("Passed:", len(passed))
print("Failed:", len(failed))
print("Blocked:", len(blocked))
print("Total:", len(tests))

for test in tests:
    print(f'{test["status"]:7} | {test["test_name"]}')

print("\nJSON report:", REPORT_JSON)
print("CSV report:", REPORT_CSV)



V2 KAFKA -> SPARK TEST
Status: BLOCKED
Topic: phase12-v2-20261017132447
Output: D:\fraud-detection-system\fraud-detection-system\reports\phase12_v2_predictions_20261017132447.csv
Spark log: D:\fraud-detection-system\fraud-detection-system\reports\phase12_v2_spark_20261017132447.log
Details: KafkaConfigurationError: KafkaConfigurationError: Unrecognized configs: {'api_version_auto_timeout_ms'}

PHASE 12 VALIDATION COMPLETE
Final status: V2_VALIDATION_BLOCKED
Passed: 18
Failed: 0
Blocked: 1
Total: 19
PASS    | artifact_presence_and_json
PASS    | phase11_decision
PASS    | feature_contract
PASS    | v2_threshold_contract
PASS    | model_artifact_loading
PASS    | model_feature_contract
PASS    | direct_v1_v2_compatibility
PASS    | api_health_v2
PASS    | api_v2_prediction_parity
PASS    | ground_truth_rejection
PASS    | request_validation
PASS    | api_performance
PASS    | fresh_process_parity
PASS    | restart_rollback_v1
PASS    | restore_v2
PASS    | kafka_schema_compatibility
PAS